# Multi-Hop Retrieval - ComplementGenerator (G+D) Evaluation (retrieval_v3)

**Before running:**
1. Settings -> Accelerator -> **T4 GPU**
2. Settings -> Internet -> **On**
3. **If re-running in an existing session: Run -> Restart & Clear Cell Outputs first**
   (clears any cached v2 modules from a prior run)
4. Google Drive `musique_data/` must contain `generator_best.pt` (required),
   `model2_best.pt` (optional), and both MuSiQue JSONL files.

**Decision gates:** `collapse_sim < 0.85`; `FULL > Graph+cos`; `FULL > 0.60`.

In [ ]:
# [EDIT IF NEEDED]
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v3"
MAX_EXAMPLES    = 300    # None for all 2417 dev queries

In [ ]:
# 1. Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 (sm_60) - change to T4 GPU")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print("CUDA OK")

In [ ]:
# 2. Clone / pull repo + install dependencies
import os
repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")
os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'sentence-transformers>=2.2.0' gdown faiss-cpu")
print("Dependencies ready")

In [ ]:
# 3. Download data + checkpoints from Google Drive
import os, gdown
download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=download_dir, quiet=False, use_cookies=False)
else:
    print("Drive data already downloaded")
print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    print(f"  {f:45s}  {os.path.getsize(f'{download_dir}/{f}')/1e6:7.1f} MB")

In [ ]:
# 4. Set up file paths - symlink data into retrieval_v2/, copy checkpoints into v3/models
import os, shutil
drive   = "/kaggle/working/drive_data"
v2_data = f"/kaggle/working/{REPO_NAME}/retrieval_v2/data/musique"
os.makedirs(v2_data,               exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",  exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",   exist_ok=True)
os.makedirs(f"{WORK_DIR}/results", exist_ok=True)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src, dst = f"{drive}/{fname}", f"{v2_data}/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")
for ckpt in ["generator_best.pt", "model2_best.pt"]:
    src, dst = f"{drive}/{ckpt}", f"{WORK_DIR}/models/{ckpt}"
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f"  Copying {ckpt} ({os.path.getsize(src)/1e6:.0f} MB)...", flush=True)
            shutil.copy(src, dst)
        print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")
    else:
        print(f"  {ckpt}: not in Drive" + ("  (REQUIRED!)" if ckpt=="generator_best.pt" else " (optional)"))
print("\nAll paths ready")

In [ ]:
# 4b. Load run_eval DIRECTLY from the v3 file (bulletproof - ignores sys.path/cache).
import os, sys, importlib.util
os.chdir(WORK_DIR)

# deps that v3 modules need: data_loader (v2), shared utils (retrieval/)
for p in [os.path.abspath(f"{WORK_DIR}/../retrieval"),
          os.path.abspath(f"{WORK_DIR}/../retrieval_v2"),
          os.path.abspath(WORK_DIR)]:
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

# purge any cached modules from a prior (v2) run
for m in ["run_eval", "model2_train", "generator_train", "data_loader"]:
    sys.modules.pop(m, None)

def load_v3(name):
    spec = importlib.util.spec_from_file_location(name, f"{WORK_DIR}/{name}.py")
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

run_eval = load_v3("run_eval")
print("run_eval loaded from:", run_eval.__file__)
assert "retrieval_v3" in run_eval.__file__, run_eval.__file__
print("OK - using retrieval_v3 run_eval")

In [ ]:
# 5. Collapse diagnostic (~5 min)
import torch, torch.nn.functional as F
from transformers import BertTokenizerFast
from generator_train import ComplementGenerator, MAX_A_LEN, MAX_B_LEN
from data_loader import load_musique, build_chain_quadruples

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_path = f"{WORK_DIR}/models/generator_best.pt"
if not os.path.exists(ckpt_path):
    print("generator_best.pt not found -- skipping collapse diagnostic")
else:
    model = ComplementGenerator().to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE), strict=False)
    model.eval()
    tok = BertTokenizerFast.from_pretrained("bert-base-uncased")
    dc, dq = load_musique(split="validation", max_examples=150, cache=False)
    quads = build_chain_quadruples(dc, dq)[:50]
    id_to_text = {c["chunk_id"]: c["text"] for c in dc}
    sims, hgf = [], []
    with torch.no_grad():
        for q in quads:
            a, b = id_to_text.get(q["chunk_a_id"], ""), id_to_text.get(q["chunk_b_pos_id"], "")
            if not a or not b:
                continue
            ea = tok(a, max_length=MAX_A_LEN, truncation=True, padding="max_length", return_tensors="pt")
            eb = tok(b, max_length=MAX_B_LEN, truncation=True, padding="max_length", return_tensors="pt")
            comp = model.extract_complement(ea["input_ids"].to(DEVICE), ea["attention_mask"].to(DEVICE), eb["input_ids"].to(DEVICE))
            bvec = model.encode_query(eb["input_ids"].to(DEVICE), eb["attention_mask"].to(DEVICE))
            sims.append(F.cosine_similarity(comp, bvec).item())
            A_sa = model.encoder1(input_ids=ea["input_ids"].to(DEVICE), attention_mask=ea["attention_mask"].to(DEVICE)).last_hidden_state[0]
            B_sa = model.encoder1(input_ids=eb["input_ids"].to(DEVICE), attention_mask=eb["attention_mask"].to(DEVICE)).last_hidden_state[0]
            ar, br = ea["attention_mask"][0].bool(), eb["attention_mask"][0].bool()
            sc = torch.matmul(B_sa[br], A_sa[ar].T) / (768 ** 0.5)
            g = 1.0 - torch.softmax(sc, dim=-1).max(dim=-1).values
            hgf.append((g > 0.70).float().mean().item())
    mc = sum(sims)/max(len(sims),1); mg = sum(hgf)/max(len(hgf),1)
    print("="*55)
    print("  COLLAPSE DIAGNOSTIC (G+D)")
    print("="*55)
    print(f"  pairs        : {len(sims)}")
    print(f"  collapse_sim : {mc:.4f}   {'OK (<0.85)' if mc < 0.85 else 'HIGH (>=0.85)'}")
    print(f"  high_g_frac  : {mg:.4f}   {'OK (>0.60)' if mg > 0.60 else 'LOW'}")
    print("="*55)

---
## Step 1: 300-Query Eval (standard, no gold edges)

In [ ]:
# 6. Standard eval (no gold edges)
assert "retrieval_v3" in run_eval.__file__, run_eval.__file__
results_standard = run_eval.main(max_examples=MAX_EXAMPLES)

---
## Step 2: Gold-Edges Oracle Eval

In [ ]:
# 7. Gold-edges oracle eval (~10 min, reuses cached embeddings)
results_gold = run_eval.main(max_examples=MAX_EXAMPLES, gold_edges=True)

In [ ]:
# 8. Final comparison table
def r10(d): return (d or {}).get("recall@10", 0)
def r5(d):  return (d or {}).get("recall@5",  0)
mdr_key  = "MDR (dense, iterative)"
cos_key  = "Graph + direct cosine (FE)"
full_key = "FULL: FE graph + complement + FAISS"
W = 60
print("="*W); print("  RESULTS SUMMARY (G+D, retrieval_v3)"); print("="*W)
_std  = globals().get("results_standard")
_gold = globals().get("results_gold")
if _std:
    print("\n  Standard eval (no gold edges):")
    print(f"    MDR           R@5={r5(_std.get(mdr_key)):.3f}  R@10={r10(_std.get(mdr_key)):.3f}")
    print(f"    Graph+cosine  R@5={r5(_std.get(cos_key)):.3f}  R@10={r10(_std.get(cos_key)):.3f}")
    print(f"    FULL          R@5={r5(_std.get(full_key)):.3f}  R@10={r10(_std.get(full_key)):.3f}")
if _gold:
    print("\n  Gold-edges oracle:")
    print(f"    Graph+cosine  R@5={r5(_gold.get(cos_key)):.3f}  R@10={r10(_gold.get(cos_key)):.3f}")
    print(f"    FULL          R@5={r5(_gold.get(full_key)):.3f}  R@10={r10(_gold.get(full_key)):.3f}")
if _std and _gold:
    full_std, full_gold = r10(_std.get(full_key)), r10(_gold.get(full_key))
    cos_std,  cos_gold  = r10(_std.get(cos_key)),  r10(_gold.get(cos_key))
    mdr_r10 = r10(_std.get(mdr_key))
    comp_vs_cos = full_std - cos_std; oracle_gain = full_gold - full_std; gold_comp = full_gold - cos_gold
    print("\n" + "="*W); print("  DECISION ANALYSIS"); print("="*W)
    print(f"  FULL vs Graph+cos (standard) : {comp_vs_cos:+.4f}  ({'helps' if comp_vs_cos>0.01 else 'marginal' if comp_vs_cos>0 else 'hurts'})")
    print(f"  FULL gold vs standard        : {oracle_gain:+.4f}  ({'coverage was bottleneck' if oracle_gain>0.03 else 'no gap'})")
    print(f"  FULL gold vs Graph+cos gold  : {gold_comp:+.4f}  ({'complement adds value' if gold_comp>0.01 else 'marginal'})")
    print(f"  FULL vs MDR                  : {full_std - mdr_r10:+.4f}  ({'beats MDR' if full_std>mdr_r10 else 'below MDR'})")
    print(f"  FULL vs v2 (0.558)           : {full_std - 0.558:+.4f}  ({'beats v2' if full_std>0.558 else 'not better'})")
    print("\n" + "="*W); print("  NEXT STEP"); print("="*W)
    if comp_vs_cos > 0.01 and gold_comp > 0.01:
        print("  Complement WORKS. G+D broke the collapse.")
    elif oracle_gain > 0.03:
        print("  Complement scores correctly when edges exist. Graph coverage is bottleneck.")
    elif full_std <= cos_std + 0.005:
        print("  Complement adds nothing -> check collapse_sim / Model 2.")
    else:
        print("  Marginal signal. Improve Model 2 query alignment.")
    print("="*W)

---
## Step 3: N_SEEDS sweep (free — finds best seeds vs hop-slots balance)

N_SEEDS controls how many of the top-10 slots are seeds vs complement-scored hops.
Lower N_SEEDS = more hop slots = more room for the complement to help.
All embeddings/edges/graph are cached, so each run is ~1-2 min (retrieval loop only).

In [ ]:
# 9. N_SEEDS sweep — find the best seeds/hops balance (cached, fast)
#    Higher FULL and bigger (FULL - cos) gap = better.
print("N_SEEDS sweep (cached):\n")
best = None
for ns in [3, 5, 7, 10]:
    run_eval.N_SEEDS = ns
    r = run_eval.main(max_examples=MAX_EXAMPLES)
    full = r["FULL: FE graph + complement + FAISS"]["recall@10"]
    cos  = r["Graph + direct cosine (FE)"]["recall@10"]
    mdr  = r["MDR (dense, iterative)"]["recall@10"]
    print(f"  N_SEEDS={ns:2d}:  FULL={full:.4f}  cos={cos:.4f}  gap={full-cos:+.4f}  (MDR={mdr:.4f})")
    if best is None or full > best[1]:
        best = (ns, full)
print(f"\n  BEST: N_SEEDS={best[0]} -> FULL={best[1]:.4f}")